In [2]:
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt

# Completeness Gaia



In [14]:
dir = '/mnt/hdcasa/splus_gaia/mc_catalogs/'

In [15]:
import pandas as pd
from pathlib import Path

dfs = [
    pd.read_csv(f)
    for f in Path(dir).glob("*.csv")
]

df_final = pd.concat(dfs, ignore_index=True)

print(df_final.shape)

(44840361, 27)


/tmp/ipykernel_13898/1129242444.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(dfs, ignore_index=True)


In [16]:
df_final = df_final[(df_final['err_mag_psf_j0660'] < 0.2)&
        (df_final['err_mag_psf_i'] < 0.2)&
        (df_final['err_mag_psf_r'] < 0.2)&
        (df_final['mag_psf_j0660']/df_final['err_mag_psf_j0660'] > 10)&
        (df_final['mag_psf_i']/df_final['err_mag_psf_i'] > 10)&
        (df_final['mag_psf_r']/df_final['err_mag_psf_r'] > 10)&
        (df_final['mag_psf_r']>= 13)&
        (df_final['mag_psf_r']<= 19.5)]

In [17]:
import pandas as pd
df = pd.DataFrame()
df["mc"] = df_final["id"].astype(str).str.extract(r"(MC\d{4})")
counts = df["mc"].value_counts().reset_index()
counts.columns = ["mc", "count"]

print(counts)

         mc   count
0    MC0064  500781
1    MC0065  496135
2    MC0063  342544
3    MC0088  339190
4    MC0087  286230
..      ...     ...
141  MC0057    5843
142  MC0052    5768
143  MC0056    5696
144  MC0055    5653
145  MC0054    5638

[146 rows x 2 columns]


In [18]:
counts.to_csv("completeness_mc_splus.csv", index=False)

In [19]:
cts = pd.read_csv("completeness_mc_gaia.csv")

In [20]:
comp = cts.merge(counts, on="mc", suffixes=("_gaia", "_splus"), how="left")

In [21]:
comp

,mc,count_gaia,count_splus
0,MC0064,496066,500781
1,MC0065,490323,496135
2,MC0063,340746,342544
3,MC0088,336843,339190
4,MC0087,283841,286230
...,...,...,...
141,MC0077,5654,5931
142,MC0052,5602,5768
143,MC0056,5536,5696
144,MC0055,5492,5653


In [22]:
len(df_final['id'].unique())

9127843

In [13]:
import splusdata

outdir = '/mnt/hdcasa/splus_gaia/mc_catalogs/'

conn = splusdata.Core()


for i in range(64, 65):
    print(f'Downloading MC{i:04d}')
    try:
        query = f"""
                select id,ra,dec,mag_psf*,err_mag_psf* from idr6.idr6 where field = 'MC{i:04d}'
            """
        mc = conn.query(
            query
        )
        mc.to_csv(f'{outdir}/MC{i:04d}.csv', index=False)
    except Exception as e:
        print(f'Error downloading MC{i:04d}: {e}')